# WinIT deep dive — delayed, distributional importance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/winit.ipynb)

**Windowed Feature Importance in Time (WinIT)** explains a prediction with in-distribution counterfactuals drawn from a reference dataset. Here we run it beside WinTSR on the exact synthetic task from the quickstart, where the true signal is known.

Papers: [WinIT, ICLR 2023](https://arxiv.org/abs/2107.14317) · [WinTSR](https://arxiv.org/abs/2412.04532)

In [ ]:
%pip install -q tslens matplotlib scikit-learn

## 1. A dataset where we know the right answer

We build 5 autocorrelated (AR(1)) input channels of length 50. The target depends on
**one feature (0), over one window (time steps 20–25)**. Everything else is noise.

That known window is our ground truth: a good interpretability method should light up
there and nowhere else.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.manual_seed(0)

SEQ_LEN, N_FEATURES = 50, 5
SIGNAL_FEATURE, SIGNAL_START, SIGNAL_END = 0, 20, 26
RHO = 0.9  # autocorrelation: real time series are not i.i.d. noise


def ar_series(n, rho=RHO):
    eps = torch.randn(n, SEQ_LEN, N_FEATURES)
    x = torch.empty_like(eps)
    x[:, 0] = eps[:, 0]
    for t in range(1, SEQ_LEN):
        x[:, t] = rho * x[:, t - 1] + (1 - rho ** 2) ** 0.5 * eps[:, t]
    return x


def make(n):
    x = ar_series(n)
    y = x[:, SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE].sum(dim=1, keepdim=True)
    return x, y


x_train, y_train = make(4000)
x_test, y_test = make(256)

ground_truth = torch.zeros(SEQ_LEN, N_FEATURES)
ground_truth[SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE] = 1

print("inputs ", tuple(x_train.shape), " targets", tuple(y_train.shape))
print(f"ground truth: feature {SIGNAL_FEATURE}, steps {SIGNAL_START}-{SIGNAL_END - 1}")

## 2. Train a small GRU

About 20 seconds on CPU.

In [ ]:
class GRUForecaster(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.gru = nn.GRU(N_FEATURES, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out.mean(dim=1))


model = GRUForecaster()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

for epoch in range(25):
    perm = torch.randperm(len(x_train))
    for i in range(0, len(x_train), 128):
        idx = perm[i : i + 128]
        opt.zero_grad()
        loss_fn(model(x_train[idx]), y_train[idx]).backward()
        opt.step()

model.eval()
with torch.no_grad():
    r2 = 1 - loss_fn(model(x_test), y_test).item() / y_test.var().item()
print(f"test R2 = {r2:.4f}   (needs to be high, or there is no signal to explain)")

## 3. How WinIT differs from WinTSR

WinTSR occludes a region with a fixed baseline and measures the output change, then rescales feature relevance with time relevance. WinIT instead moves backward from the most recent time step, progressively replacing each feature with **counterfactual samples drawn from real reference data**. It measures the distributional change after every replacement and differences consecutive scores to isolate each time step's marginal contribution.

Because this is forecasting rather than classification, WinIT uses prediction difference (`pd`); classification uses Jensen–Shannon (`js`). WinIT still returns `(batch, n_output, seq_len, n_features)`. Its “delay” is baked into progressive backward masking and the distributional-distance calculation—it is not an extra output axis.

## 4. Explain the same test samples

WinTSR receives the quickstart's zero baseline. WinIT instead receives `x_train` as its reference dataset, so sampled replacements remain plausible under the training distribution. WinIT's forecasting formatter expects `(batch, pred_len, features)`, so the adapter below adds a singleton feature axis to this quickstart model's `(batch, 1)` output without changing its predictions.

In [ ]:
import argparse

from tslens import WinTSR
from tslens.attr import WinIT

inputs = x_test[:16]
baselines = torch.zeros_like(inputs)

wintsr_attr = WinTSR(model).attribute(
    inputs, baselines=baselines, threshold=0.5
)

def winit_forward(x):
    return model(x).unsqueeze(-1)

args = argparse.Namespace(
    seq_len=SEQ_LEN, task_name="forecast", pred_len=1, features="S", seed=0,
)
np.random.seed(0)
winit = WinIT(winit_forward, data=x_train, args=args)
winit_attr = winit.attribute(
    inputs=inputs,
    additional_forward_args=None,
    attributions_fn=abs,
)

print("WinTSR:", tuple(wintsr_attr.shape))
print("WinIT:  ", tuple(winit_attr.shape))

## 5. Compare the saliency maps

Each attribution map is averaged over the same 16 test samples and output dimension. The planted signal occupies feature 0, steps 20–25.

In [ ]:
import matplotlib.pyplot as plt

wintsr_saliency = wintsr_attr.abs().mean(dim=(0, 1)).detach()
winit_saliency = winit_attr.abs().mean(dim=(0, 1)).detach()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))

axes[0].imshow(ground_truth.T, aspect="auto", cmap="Greys", vmin=0, vmax=1)
axes[0].set_title("Ground truth")
axes[1].imshow(wintsr_saliency.T, aspect="auto", cmap="viridis")
axes[1].set_title("WinTSR saliency")
axes[2].imshow(winit_saliency.T, aspect="auto", cmap="viridis")
axes[2].set_title("WinIT saliency")

for ax in axes:
    ax.set_xlabel("time step")
    ax.set_ylabel("feature")
    ax.set_yticks(range(N_FEATURES))

plt.tight_layout()
plt.show()

## 6. Score both against the ground truth

Average precision compares every sample's saliency map with the known mask. Random guessing scores about 0.024 because the signal covers 6 of 250 cells.

In [ ]:
from sklearn.metrics import average_precision_score


def average_precision(attr):
    a = attr.abs()
    if a.dim() == 4:
        a = a.mean(dim=1)
    rows = a.reshape(len(a), -1).detach().numpy()
    truth = ground_truth.reshape(-1).numpy()
    return float(np.mean([average_precision_score(truth, r) for r in rows]))


scores = {
    "WinTSR": average_precision(wintsr_attr),
    "WinIT": average_precision(winit_attr),
    "Random baseline": float(ground_truth.mean()),
}

print(f"{'method':<18}{'avg precision':>14}")
print("-" * 32)
for name, score in scores.items():
    print(f"{name:<18}{score:>14.4f}")

## 7. When to reach for WinIT

Choose WinIT when:

- you are explaining a classification task and want its Jensen–Shannon distributional metric;
- in-distribution counterfactuals are more meaningful than a fixed zero or mean baseline; or
- “how much does forgetting this time step hurt the prediction right now?” is more natural than “what if this time step were absent?”

WinTSR remains a strong default when you want a direct, baseline-controlled perturbation explanation with explicit time and feature rescaling.

## Next steps

- Try a classification model to switch WinIT from prediction difference to Jensen–Shannon divergence.
- Replace `x_train` with another representative reference set and compare how the counterfactual distribution changes the result.
- See the [methods guide](https://khairulislam.github.io/tslens/methods/) and [integration cookbook](https://khairulislam.github.io/tslens/integration/) for more comparisons.

```bibtex
@inproceedings{leung2023temporal,
  title={Temporal Dependencies in Feature Importance for Time Series Predictions},
  author={Leung, Kin Kwan and Rooke, Clayton and Smith, Jonathan and Zuberi, Saba and Volkovs, Maksims},
  booktitle={International Conference on Learning Representations},
  year={2023}
}

@article{islam2024wintsr,
  title={WinTSR: A Windowed Temporal Saliency Rescaling Method for Interpreting Time Series Deep Learning Models},
  author={Islam, Md Khairul and Fox, Judy},
  journal={arXiv preprint arXiv:2412.04532},
  year={2024}
}
```